# Becke Partition 二阶梯度简单理解

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol)
    grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids.build(sort_grids=False)
    return mol, grids

## PySCF 数值二阶梯度导数

目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。但需要留意，由于 PySCF 目前没有实现二阶解析格点导数，我们有必要自己实现一版。为此，我们需要比较完善的函数，计算所有二阶梯度；而不是像一阶梯度那样只需要稍微验证一下。

In [4]:
xyz_orig = np.array([[0.0, 0.0, 0.0], [1.0, 0.1, 0.2], [0.3, 1.1, 0.2], [0.1, 0.1, 1.2]])  # in angstrom

def perturb_xyz(xyz, A, t, sgn, delta):
    xyz_pert = xyz.copy()
    xyz_pert[A, t] += sgn * delta
    return xyz_pert

def perturb_mol(A, t, sgn, delta):
    xyz_coords = perturb_xyz(xyz_orig, A, t, sgn, delta)
    atm_symbols = ["N", "H", "H", "H"]
    xyz_str = "\n".join(f"{atm} {x:.6f} {y:.6f} {z:.6f}" for atm, (x, y, z) in zip(atm_symbols, xyz_coords))
    mol_pert = gto.Mole(atom=xyz_str, basis="def2-TZVP", max_memory=32000).build()
    return mol_pert

def perturb_mol_grids(A, t, sgn, delta):
    mol_pert = perturb_mol(A, t, sgn, delta)
    grids_pert = dft.grid.Grids(mol_pert)
    grids_pert.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids_pert.build(sort_grids=False)
    return mol_pert, grids_pert

首先我们拿到没有坐标微扰的分子与格点。

In [5]:
mol, grids = perturb_mol_grids(A=0, t=0, sgn=1, delta=0.0)

In [6]:
nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]

natm = atm_coords.shape[0]
ngrids = grid_coords.shape[0]
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)

微扰的分子与格点放在一个字典中。

In [7]:
interval = 3e-5
dict_perturb = {(A, t, sgn): perturb_mol_grids(A, t, sgn, delta=interval) for A in range(len(xyz_orig)) for t in range(3) for sgn in [-1, 1]}
dict_w = {(A, t, sgn): grids_pert.weights for (A, t, sgn), (_, grids_pert) in dict_perturb.items()}
dict_dw = {(A, t, sgn): hessian.rks.get_dweight_dA(mol_pert, grids_pert) for (A, t, sgn), (mol_pert, grids_pert) in dict_perturb.items()}

我们可以先验证一下一阶梯度量的数值导数是否正确。

In [8]:
dw_ndiff = np.zeros((natm, 3, ngrids))
for A in range(natm):
    for t in range(3):
        dw_plus = dict_w[(A, t, 1)][..., nonpad_mask]
        dw_minus = dict_w[(A, t, -1)][..., nonpad_mask]
        dw_ndiff[A, t] = (dw_plus - dw_minus) / (2 * interval / data.nist.BOHR)
assert np.allclose(dw_ndiff, hessian.rks.get_dweight_dA(mol, grids)[..., nonpad_mask])

二阶梯度量可以通过一阶梯度量的数值导数来计算。

In [9]:
ddw_ndiff = np.zeros((natm, 3, natm, 3, ngrids))
for A in range(natm):
    for t in range(3):
        dw_plus = dict_dw[(A, t, 1)][..., nonpad_mask]
        dw_minus = dict_dw[(A, t, -1)][..., nonpad_mask]
        ddw_ndiff[A, t] = (dw_plus - dw_minus) / (2 * interval / data.nist.BOHR)

我们可以通过检查对称性来简单地从一个角度验证正确性，以及估算数值误差大小。下面的误差估计是比较保守的。

In [10]:
assert np.allclose(ddw_ndiff, ddw_ndiff.transpose(2, 3, 0, 1, 4), rtol=1e-4, atol=5e-7)  # Check symmetry

最后我们对称化该数值梯度，作为参考值。

- `ddw_ref` 二阶格点权重梯度，维度 $(A, t, B, s, g)$ `(natm, 3, natm, 3, ngrids)`。

    $$
    \frac{\partial^2 w_g}{\partial R_{At} \partial R_{Bs}}
    $$

In [11]:
ddw_ref = 0.5 * (ddw_ndiff + ddw_ndiff.transpose(2, 3, 0, 1, 4))

## Becke Partition 二阶梯度实现与公式对应

我们需要再来一次。

### 1. 梯度无关量

- `wquad` $w_g^\text{quad}$：原始 Lebedev 权重，维度 $(g,)$ `(ngrids,)`

- `a` $a_{AB}$：Becke radii 矫正表，维度 $(A, B)$ `(natm, natm)`

In [12]:
wquad = quadrature_weights
a = radii_table

### 2. 原子间距离 $\Vert R \Vert_{AB}$

- `atm_coords` $R_{A t}$：原子坐标，维度 $(A, 3)$ `(natm, 3)`

- `atom_dist` $\Vert R \Vert_{AB}$：原子间距离，维度 $(A, B)$ `(natm, natm)`；其只作为分母出现，对角元设为 $\infty$，避免除零

    $$
    \Vert R \Vert_{AB} = 
    \begin{cases}
    \sqrt{\sum_t (R_{B t} - R_{A t})^2} & A \neq B \\
    \infty & A = B
    \end{cases}
    $$

In [13]:
atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)
for i in range(natm):
    atom_dist[i, i] = np.inf

- $\Vert \partial R \Vert_{A B t}$ `dR_atom_dist` 原子距离导数，导数只对原子 $A$ 进行，维度 $(A, B, t)$ `(natm, natm, 3)`：

    $$
    \Vert \partial R \Vert_{A B t} := \frac{\partial \Vert R \Vert_{AB}}{\partial R_{A t}} = \frac{R_{A t} - R_{B t}}{\Vert R \Vert_{AB}}
    $$

In [14]:
dR_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]
assert np.allclose(- dR_atom_dist.swapaxes(0, 1), dR_atom_dist)

在处理原子距离的二阶导数前，我们重新回顾一下一阶导数。我们之所以要用 $\Vert \partial R \Vert_{A B t}$ 这种不二不三的形式，是因为严格来说 $B = A$ 的导数情况我们没有考虑。但对于当前的原子距离问题，$B = A$ 的情况是没有意义的 (同原子没有贡献)。
$$
\begin{align*}
\frac{\partial \Vert R \Vert_{M N}}{\partial R_{A t}} &= \delta_{A M} \frac{\partial \Vert R \Vert_{M N}}{\partial R_{M t}} + \delta_{A N} \frac{\partial \Vert R \Vert_{M N}}{\partial R_{N t}} \quad (M \text{ unrelated to } N) \\
&= \delta_{A M} \Vert \partial R \Vert_{A N t} - \delta_{A N} \Vert \partial R \Vert_{A M t} \\
\end{align*}
$$

### 记号约定回顾与二阶导数记号

本节沿用 `10-2-*.ipynb` 的记号：下标 $A, B$ 作导数原子对应 $R_{At}, R_{Bs}$；$M, N$ 作 $\mu_{MNg}$ 的两端点；$A_g$ 为格点 $g$ 所属原子 (`atm_indices[g]`)。格点坐标 $r_g$ 暂视为不依赖 $R_A$ (全导数问题留至最后一步处理)。并补充二阶导数记号：

- `dR_*` / `dmu_*`：一阶偏导 $\partial/\partial R_{At}$ / $\partial/\partial\mu_{MNg}$。
- `dRdR_*` / `dmu_dmu_*`：二阶偏导 $\partial^2/(\partial R_{At}\partial R_{Bs})$ / $\partial^2/\partial\mu^2$。
- $\mu_{MNg}$ 的二阶坐标导数有四种 role 组合 (两个导数各自可落在 $M$ 或 $N$ 上)：
  - `dRdR_mu_roleAA` $=\partial^2\mu_{MNg}/\partial R_{Mt}\partial R_{Ms}$ (均对左端点 $M$)
  - `dRdR_mu_roleAB` $=\partial^2\mu_{MNg}/\partial R_{Mt}\partial R_{Ns}$ (一阶对 $M$、二阶对 $N$)
  - `dRdR_mu_roleBB` $=\partial^2\mu_{MNg}/\partial R_{Nt}\partial R_{Ns}$ (均对右端点 $N$)
  - role BA $=$ role AB 在 $(t,s)$ 上的转置：$\partial^2\mu_{MNg}/\partial R_{Nt}\partial R_{Ms}=$ `dRdR_mu_roleAB`[:,:,s,t,:]。

连乘积 $P_{Mg}=\prod_{N\neq M}s_{MNg}$ 的导数用对数化简：

- `dR_log_P` $\partial\log P_{Mg}/\partial R_{At}$ 一阶对数导数，维度 $(M, A, t, g)$ `(natm, natm, 3, ngrids)`；
- `dRdR_log_P` $\partial^2\log P_{Mg}/\partial R_{At}\partial R_{Bs}$ 二阶对数导数，维度 $(M, A, t, B, s, g)$ `(natm, natm, 3, natm, 3, ngrids)`。

则 $\partial^2 P_{Mg}/\partial R_{At}\partial R_{Bs}=P_{Mg}\big(\texttt{dRdR\_log\_P}+\texttt{dR\_log\_P}\otimes\texttt{dR\_log\_P}\big)$。对数导数直接由 role 结构给出，避免对 $P_{Mg}$ 作除法 (当 $P_{Mg}$ 很小时仍能保持精度)。

### 椭球坐标与零阶权重 (10-2 回顾)

以下为零阶量，记号与公式完全沿用 10-2：

- `grid_dist` $\Vert r\Vert_{Ag}$ 原子与格点间距离，维度 $(A, g)$ `(natm, ngrids)`。
- `mu` $\mu_{MNg}$ 椭球坐标差分量，维度 $(M, N, g)$ `(natm, natm, ngrids)`：

    $$
    \mu_{MNg}=\frac{\Vert r\Vert_{Mg}-\Vert r\Vert_{Ng}}{\Vert R\Vert_{MN}} \tag{11}
    $$

- `s` $s_{MNg}$ Becke 特征函数 (对角 $M=N$ 置 $1$)，维度 $(M, N, g)$ `(natm, natm, ngrids)`；`P` $P_{Mg}=\prod_{N\neq M}s_{MNg}$，维度 $(M, g)$ `(natm, ngrids)`；`Z` $Z_g=\sum_M P_{Mg}$，维度 $(g,)$ `(ngrids,)`。
- `Pg` $P^g_{A_g}=P_{A_g g}$ (格点所属原子的权重，按 $A_g$ 收集)，维度 $(g,)$ `(ngrids,)`；`w` $w_g=w_g^{\text{quad}}\,P^g_{A_g}/Z_g$，维度 $(g,)$ `(ngrids,)`。

In [15]:
# zero-order quantities (recap of 10-2)
grid_dist = np.linalg.norm(grid_coords[None, :, :] - atm_coords[:, None, :], axis=-1)  # (A, g)
mu = (grid_dist[:, None, :] - grid_dist[None, :, :]) / atom_dist[:, :, None]  # eq (11), (M, N, g)

# switch function closures (from 10-2)
fn_p = lambda nu: 1.5 * nu - 0.5 * nu**3
fn_f1 = fn_p
fn_f2 = lambda nu: fn_p(fn_f1(nu))
fn_f3 = lambda nu: fn_p(fn_f2(nu))
fn_s3 = lambda nu: 0.5 * (1 - fn_f3(nu))
fn_nu = lambda mu, a: mu + a * (1 - mu**2)
fn_s = lambda mu, a: fn_s3(fn_nu(mu, a))

s = fn_s(mu, a[:, :, None])
for M in range(natm):
    s[M, M] = 1
P = s.prod(axis=1)  # (M, g)
Z = P.sum(axis=0)
Pg = P[atm_indices, np.arange(ngrids)]
w = wquad * Pg / Z
assert np.allclose(w, grids.weights[nonpad_mask])

### 开关函数 $s(\mu)$ 的一阶与二阶导数

沿用 $\nu=\mu+a(1-\mu^2)$，$s=\frac12(1-f_3(\nu))$，$f_3=p\circ p\circ p$，$p(x)=\frac32x-\frac12x^3$。一阶导数 (10-2)：
$$
p'=\tfrac32(1-\nu^2),\quad f_3'=p'(f_2)\,p'(f_1)\,p'(\nu),\quad s'_{\nu}=-\tfrac12 f_3',\quad \nu'=1-2a\mu,\quad s'=s'_{\nu}\,\nu'.
$$
二阶导数：记 $g_0=p'(\nu),\,g_1=p'(f_1),\,g_2=p'(f_2)$，$p''(x)=-3x$，则
$$
f_3''=-3\big[f_2\,(g_1 g_0)^2+f_1\,g_2\,g_0^2+\nu\,g_2\,g_1\big],\qquad
\frac{d^2 s}{d\mu^2}=-\tfrac12 f_3''\,(\nu')^2-\tfrac12 f_3'\,\nu'',\qquad \nu''=-2a.
$$
对应对数导数：

- $\partial s_{MNg}/\partial\mu_{MNg}$ `dmu_s` 开关函数一阶导，维度 $(M, N, g)$ `(natm, natm, ngrids)`；
- $\partial\log s_{MNg}/\partial\mu_{MNg}=s'/s$ `dmu_log_s` 一阶对数导数，维度 $(M, N, g)$ `(natm, natm, ngrids)`；
- $\partial^2\log s_{MNg}/\partial\mu_{MNg}^2=s''/s-(s'/s)^2$ `dmu_dmu_log_s` 二阶对数导数，维度 $(M, N, g)$ `(natm, natm, ngrids)`。

当 $|s|<10^{-14}$ 时将上述对数导数置 $0$ (避免 $s\to0$ 处除零发散)；对角 $M=N$ 亦置 $0$。

In [16]:
# first derivative of s wrt mu (10-2)
fn_d_p = lambda nu: 1.5 * (1 - nu**2)
fn_d_f1 = fn_d_p
fn_d_f2 = lambda nu: fn_d_p(fn_f1(nu)) * fn_d_f1(nu)
fn_d_f3 = lambda nu: fn_d_p(fn_f2(nu)) * fn_d_f2(nu)
fn_d_s3 = lambda nu: -0.5 * fn_d_f3(nu)
fn_d_nu = lambda mu, a: 1 - 2 * a * mu
fn_d_s = lambda mu, a: fn_d_s3(fn_nu(mu, a)) * fn_d_nu(mu, a)
dmu_s = fn_d_s(mu, a[:, :, None])  # s'(mu), (M, N, g)

s_safe_mask = np.abs(s) > 1e-14
s_safe = s.copy(); s_safe[~s_safe_mask] = 1.0
dmu_log_s = dmu_s / s_safe
for M in range(natm):
    dmu_log_s[M, M] = 0.0

# second derivative of s wrt mu: s''(mu) = s3''(nu) (nu')^2 + s3'(nu) nu''
def fn_d_d_s3(nu):
    g0 = fn_d_p(nu); g1 = fn_d_p(fn_f1(nu)); g2 = fn_d_p(fn_f2(nu))
    f1 = fn_f1(nu); f2 = fn_f2(nu)
    return -0.5 * (-3.0 * (f2 * (g1 * g0)**2 + f1 * g2 * g0**2 + nu * g2 * g1))
fn_d_d_nu = lambda mu, a: -2 * a
fn_d_d_s = lambda mu, a: fn_d_d_s3(fn_nu(mu, a)) * fn_d_nu(mu, a)**2 + fn_d_s3(fn_nu(mu, a)) * fn_d_d_nu(mu, a)
dmu_dmu_s = fn_d_d_s(mu, a[:, :, None])  # s''(mu), (M, N, g)
# d2 log s / dmu2 = s''/s - (s'/s)^2
dmu_dmu_log_s = np.where(s_safe_mask, dmu_dmu_s / s_safe, 0.0) - dmu_log_s**2
for M in range(natm):
    dmu_dmu_log_s[M, M] = 0.0

### 椭球坐标 $\mu_{MNg}$ 的一阶与二阶坐标导数

记 $f=\Vert r\Vert_{Mg}-\Vert r\Vert_{Ng}$，$g=\Vert R\Vert_{MN}$，$\mu=f/g$；$\hat r_A=(R_A-r_g)/\Vert r\Vert_{Ag}$，$\hat R_{AB}=(R_A-R_B)/\Vert R\Vert_{AB}$。一阶 (10-2)：
$$
\partial_{R_M}\mu=\frac{\hat r_A-\mu\hat R_{AB}}{\Vert R\Vert_{MN}}\;(\text{role A}),\qquad
\partial_{R_N}\mu=\frac{-\hat r_B+\mu\hat R_{AB}}{\Vert R\Vert_{MN}}\;(\text{role B}).
$$
对应变量 (一阶，10-2 回顾)：

- $\Vert\partial r\Vert_{Atg}:=\partial\Vert r\Vert_{Ag}/\partial R_{At}=\hat r_A$ `dR_grid_dist` 格点距离导数，维度 $(A, t, g)$ `(natm, 3, ngrids)`；
- $\partial\mu_{MNg}/\partial R_{Mt}$ (role A) `dR_mu_roleA`，维度 $(M, N, t, g)$ `(natm, natm, 3, ngrids)`；
- $\partial\mu_{MNg}/\partial R_{Nt}$ (role B) `dR_mu_roleB`，维度同上 (两者反对称：`dR_mu_roleA`$=-$`dR_mu_roleB`.swapaxes(0,1))。

二阶由商法则 $\partial_{ij}(f/g)=\big[f_{ij}g-(f_i g_j+g_i f_j)-f g_{ij}\big]/g^2+2f g_i g_j/g^3$ 给出。各 role 的 $(f_X, g_X, f_{XY}, g_{XY})$ 取值如下 (记 $\mathrm{Proj}(\hat v)=I-\hat v\hat v^T$)：

| role | $f_X$ | $g_X$ | $f_{XY}$ | $g_{XY}$ |
|---|---|---|---|---|
| AA | $\hat r_A$ | $\hat R_{AB}$ | $\mathrm{Proj}(\hat r_A)/\Vert r\Vert_{Ag}$ | $\mathrm{Proj}(\hat R_{AB})/\Vert R\Vert_{MN}$ |
| AB | $f_X=\hat r_A,\ f_Y=-\hat r_B$ | $g_X=\hat R_{AB},\ g_Y=-\hat R_{AB}$ | $0$ | $-\mathrm{Proj}(\hat R_{AB})/\Vert R\Vert_{MN}$ |
| BB | $-\hat r_B$ | $-\hat R_{AB}$ | $-\mathrm{Proj}(\hat r_B)/\Vert r\Vert_{Bg}$ | $\mathrm{Proj}(\hat R_{AB})/\Vert R\Vert_{MN}$ |

对应变量 (二阶，维度均为 $(M, N, t, s, g)$ `(natm, natm, 3, 3, ngrids)`)：

- $\partial^2\mu_{MNg}/\partial R_{Mt}\partial R_{Ms}$ `dRdR_mu_roleAA` (均对左端点 $M$)；
- $\partial^2\mu_{MNg}/\partial R_{Mt}\partial R_{Ns}$ `dRdR_mu_roleAB` (一阶对 $M$、二阶对 $N$)，role BA $=$ `dRdR_mu_roleAB` 在 $(t,s)$ 转置；
- $\partial^2\mu_{MNg}/\partial R_{Nt}\partial R_{Ns}$ `dRdR_mu_roleBB` (均对右端点 $N$)。

(对角 $M=N$ 处 $\Vert R\Vert_{MM}=\infty$，导数无定义，置 $0$。)

In [17]:
# first derivative of mu wrt atomic coords (10-2)
dR_grid_dist = (atm_coords[:, :, None] - grid_coords.T[None, :, :]) / grid_dist[:, None, :]  # (A, t, g)
dR_mu_roleA = ( dR_grid_dist[:, None, :, :] - mu[:, :, None, :] * dR_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]
dR_mu_roleB = (-dR_grid_dist[None, :, :, :] + mu[:, :, None, :] * dR_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]

# second derivative of mu wrt atomic coords (vectorized, 4 role blocks)
# note: diagonal M=N has atom_dist = inf, producing nan in d2mu; suppressed here and zeroed below
uA = dR_grid_dist[:, None, :, :]      # (M, N, t, g) = hat r_A for pair (M, N)
uB = dR_grid_dist[None, :, :, :]      # (M, N, t, g) = hat r_B for pair (M, N) = dR_grid_dist[N]
U  = dR_atom_dist[:, :, :, None]      # (M, N, t, g) = hat R_{MN}, broadcast over g
f_ab = grid_dist[:, None, :] - grid_dist[None, :, :]   # (M, N, g) = f = r_A - r_B
eye3 = np.eye(3)
def proj(v):  # Proj(v) = I - v v^T, outer over (t, s) with g shared
    return eye3[None, None, :, :, None] - v[:, :, :, None, :] * v[:, :, None, :, :]
PuA, PuB, PU = proj(uA), proj(uB), proj(U)
Rn5 = atom_dist[:, :, None, None, None]    # (M, N, 1, 1, 1) = g = |R_{MN}|
f5 = f_ab[..., None, None, :]               # (M, N, 1, 1, g) = f
def d2mu(fX, fY, fXY, gX, gY, gXY):
    ofg = fX[:,:,:,None,:]*gY[:,:,None,:,:] + gX[:,:,:,None,:]*fY[:,:,None,:,:]  # f_X g_Y + g_X f_Y
    ogg = gX[:,:,:,None,:]*gY[:,:,None,:,:]                                       # g_X g_Y
    return (fXY*Rn5 - ofg - f5*gXY)/Rn5**2 + 2*f5*ogg/Rn5**3
with np.errstate(invalid="ignore", divide="ignore"):
    dRdR_mu_roleAA = d2mu(uA,  uA,  PuA / grid_dist[:, None, None, None, :],   U,  U,  PU/Rn5)
    dRdR_mu_roleAB = d2mu(uA, -uB,  np.zeros_like(PuA),                       U, -U, -PU/Rn5)
    dRdR_mu_roleBB = d2mu(-uB, -uB, -PuB / grid_dist[None, :, None, None, :], -U, -U,  PU/Rn5)
for A in range(natm):  # diagonal M=N is undefined (atom_dist = inf); zero out
    dRdR_mu_roleAA[A, A] = 0; dRdR_mu_roleAB[A, A] = 0; dRdR_mu_roleBB[A, A] = 0

### 权重 $P_{Mg}$ 与归一化 $Z_g$ 的二阶导数

一阶导数 (10-2)：`dR_P` $\partial P_{Mg}/\partial R_{At}=P_{Mg}\,\texttt{dR\_log\_P}$，维度 $(M, A, t, g)$ `(natm, natm, 3, ngrids)`；`dR_Z` $\partial Z_g/\partial R_{At}=\sum_M\texttt{dR\_P}$，维度 $(A, t, g)$ `(natm, 3, ngrids)`；`dR_Pg` 为按 $A_g$ 收集的 $\partial P_{A_g}/\partial R_{At}$，维度 $(A, t, g)$ `(natm, 3, ngrids)`。

二阶导数用对数法。对 $\ln P_{Mg}=\sum_{N\neq M}\ln s_{MNg}$ 求导：
$$
\texttt{dRdR\_log\_P}_{MAtBsg}=\sum_{N\neq M}\frac{\partial^2\ln s_{MNg}}{\partial R_{At}\partial R_{Bs}},\qquad
\frac{\partial^2\ln s_{MNg}}{\partial R_{At}\partial R_{Bs}}=w_{MNg}\,(\partial_A\mu_{MNg})(\partial_B\mu_{MNg})+t_{MNg}\,\partial_{AB}\mu_{MNg},
$$
其中 $w=$ `dmu_dmu_log_s`，$t=$ `dmu_log_s`。对固定 $M$，$(A,B)$ 仅在 $\{M,N\}$ 的四种 role 组合处非零，故 `dRdR_log_P` 在 $(A,B)$ 上稀疏：

- $(A,B)=(M,M)$：role AA，对 $N$ 求和；
- $(A,B)=(M,N)$：role AB；$(A,B)=(N,M)$：role BA；$(A,B)=(N,N)$：role BB (后三者每对单独，不求和)。

最后
$$
\frac{\partial^2 P_{Mg}}{\partial R_{At}\partial R_{Bs}}=P_{Mg}\big(\texttt{dRdR\_log\_P}+\texttt{dR\_log\_P}\otimes\texttt{dR\_log\_P}\big),\qquad
\frac{\partial^2 Z_g}{\partial R_{At}\partial R_{Bs}}=\sum_M\frac{\partial^2 P_{Mg}}{\partial R_{At}\partial R_{Bs}}.
$$
对应变量：

- `dRdR_log_P` $\partial^2\log P_{Mg}/\partial R_{At}\partial R_{Bs}$，维度 $(M, A, t, B, s, g)$ `(natm, natm, 3, natm, 3, ngrids)`；
- `dRdR_P` $\partial^2 P_{Mg}/\partial R_{At}\partial R_{Bs}$，维度同上；
- `dRdR_Z` $\partial^2 Z_g/\partial R_{At}\partial R_{Bs}$，维度 $(A, t, B, s, g)$ `(natm, 3, natm, 3, ngrids)`；
- `dRdR_Pg` $\partial^2 P^g_{A_g}/\partial R_{At}\partial R_{Bs}$ (按 $A_g$ 收集)，维度 $(A, t, B, s, g)$ `(natm, 3, natm, 3, ngrids)`。

In [18]:
# first derivatives of P, Z (recap of 10-2)
dR_P_roleA = np.einsum("Ag, ANg, ANtg -> Atg", P, dmu_log_s, dR_mu_roleA)
dR_P_roleB = np.einsum("Mg, MAg, MAtg -> MAtg", P, dmu_log_s, dR_mu_roleB)
dR_P = dR_P_roleB.copy()
for A in range(natm):
    dR_P[A, A] = dR_P_roleA[A]
dR_Z = dR_P.sum(axis=0)
dR_Pg = dR_P[atm_indices, :, :, np.arange(ngrids)].transpose(1, 2, 0)

In [19]:
# log first derivative dR_log_P[M, A, t, g] = d log P_M / d R_At
# (computed directly from role structure, NOT as dR_P / P, to keep precision when P_M is tiny)
dR_log_P_roleA = np.einsum("MNg, MNtg -> Mtg", dmu_log_s, dR_mu_roleA)   # role A: A = M
dR_log_P_roleB = dmu_log_s[:, :, None, :] * dR_mu_roleB                  # role B: A = N, (M, N, t, g)
dR_log_P = dR_log_P_roleB.copy()
for M in range(natm):
    dR_log_P[M, M] = dR_log_P_roleA[M]

# log second derivative dRdR_log_P[M, A, t, B, s, g] = d2 log P_M / dR_A dR_B
#   = sum_{N!=M} [ w (d_A mu)(d_B mu) + t d2_AB mu ],  sparse over (A,B) in {M,N}
dRdR_mu_roleBA = dRdR_mu_roleAB.transpose(0, 1, 3, 2, 4)   # role BA = role AB transposed in (t, s)
L2_AA = (np.einsum("MNg, MNtg, MNsg -> Mtsg", dmu_dmu_log_s, dR_mu_roleA, dR_mu_roleA)
       + np.einsum("MNg, MNtsg -> Mtsg", dmu_log_s, dRdR_mu_roleAA))            # (M, t, s, g), summed over N
L2_AB = (np.einsum("MNg, MNtg, MNsg -> MNtsg", dmu_dmu_log_s, dR_mu_roleA, dR_mu_roleB)
       + np.einsum("MNg, MNtsg -> MNtsg", dmu_log_s, dRdR_mu_roleAB))           # (M, N, t, s, g)
L2_BA = (np.einsum("MNg, MNtg, MNsg -> MNtsg", dmu_dmu_log_s, dR_mu_roleB, dR_mu_roleA)
       + np.einsum("MNg, MNtsg -> MNtsg", dmu_log_s, dRdR_mu_roleBA))
L2_BB = (np.einsum("MNg, MNtg, MNsg -> MNtsg", dmu_dmu_log_s, dR_mu_roleB, dR_mu_roleB)
       + np.einsum("MNg, MNtsg -> MNtsg", dmu_log_s, dRdR_mu_roleBB))
# scatter into dRdR_log_P[M, A, t, B, s, g]; diagonal (m,m,m) is shared by all 4 roles ->
# use += (role AA contributes the sum there; BB/AB/BA contribute 0 since w,t have zero diagonal)
dRdR_log_P = np.zeros((natm, natm, 3, natm, 3, ngrids))
idx = np.arange(natm); Mi, Ni = np.indices((natm, natm)); mi, ni = Mi.ravel(), Ni.ravel()
dRdR_log_P[idx, idx, :, idx, :, :] += L2_AA                                       # role AA: (A,B)=(M,M)
dRdR_log_P[mi, ni, :, ni, :, :] += L2_BB.reshape(natm*natm, 3, 3, ngrids)         # role BB: (N,N)
dRdR_log_P[mi, mi, :, ni, :, :] += L2_AB.reshape(natm*natm, 3, 3, ngrids)          # role AB: (M,N)
dRdR_log_P[mi, ni, :, mi, :, :] += L2_BA.reshape(natm*natm, 3, 3, ngrids)          # role BA: (N,M)

In [20]:
# d2 P_M = P_M * (dRdR_log_P + dR_log_P_A ⊗ dR_log_P_B)
dRdR_P = P[:, None, None, None, None, :] * (
    dRdR_log_P + dR_log_P[:, :, :, None, None, :] * dR_log_P[:, None, None, :, :, :])   # (M, A, t, B, s, g)
dRdR_Z = dRdR_P.sum(axis=0)   # (A, t, B, s, g)
# gather P_{A_g}: dRdR_Pg[A, t, B, s, g] = dRdR_P[atm_indices[g], A, t, B, s, g]
dRdR_Pg = dRdR_P[atm_indices, :, :, :, :, np.arange(ngrids)].transpose(1, 2, 3, 4, 0)

### 格点权重 $w_g$ 的二阶导数

记 $q_g=P_{A_g}/Z_g$ ($w_g=w_g^{\text{quad}}\,q_g$)。先用不考虑 $r_g$ 依赖的偏导给出 `ddw_partial` (商法则)：
$$
\frac{\partial^2 q_g}{\partial R_{At}\partial R_{Bs}}
=\frac{\partial_{AB}P_{A_g}-(\partial_B q_g)(\partial_A Z_g)-q_g\,\partial_{AB}Z_g}{Z_g}
-\frac{(\partial_A q_g)(\partial_B Z_g)}{Z_g}.
$$
最后处理 $r_g$ 对 $R_{A_g}$ 的依赖。对 $A,B\neq A_g$，偏导即全导 (正确)；对 $A=A_g$ 或 $B=A_g$，偏导错误，由平移不变性 $\sum_A\partial_{AB}w_g=0$ (对 $A,B$ 两个指标均成立) 补齐：
$$
\partial_{A_g B}w_g=-\sum_{A\neq A_g}\partial_{AB}w_g,\qquad
\partial_{A A_g}w_g=-\sum_{B\neq A_g}\partial_{AB}w_g,\qquad
\partial_{A_g A_g}w_g=\sum_{A\neq A_g,\,B\neq A_g}\partial_{AB}w_g.
$$
注意求和须排除 $A_g$ 本身 (其偏导错误)，角点 $(A_g,A_g)$ 为双重求和。

- `ddw` $\partial^2 w_g/\partial R_{At}\partial R_{Bs}$ 二阶格点权重导数，维度 $(A, t, B, s, g)$ `(natm, 3, natm, 3, ngrids)`。

In [21]:
# quotient rule (partial, r_g treated as fixed)
q = Pg / Z
dq = (dR_Pg - q * dR_Z) / Z   # (A, t, g), first derivative of q = P_{A_g} / Z
term1 = np.einsum("Bsg, Atg -> AtBsg", dq, dR_Z)   # (dq_B)(dZ_A)
term2 = np.einsum("Atg, Bsg -> AtBsg", dq, dR_Z)   # (dq_A)(dZ_B)
d2q = (dRdR_Pg - term1 - q * dRdR_Z) / Z - term2 / Z
ddw_partial = wquad * d2q   # (A, t, B, s, g), partial (r_g fixed)

# translation invariance: fill A=A_g and B=A_g blocks.
#   - sums exclude A_g (whose partial is wrong);
#   - corner (A_g, A_g) = double sum over A!=A_g, B!=A_g.
ddw = ddw_partial.copy()
for g in range(ngrids):
    Ag = int(atm_indices[g])
    sumB_ne = ddw_partial[:, :, :, :, g].sum(axis=2) - ddw_partial[:, :, Ag, :, g]   # sum over B != A_g
    sumA_ne = ddw_partial[:, :, :, :, g].sum(axis=0) - ddw_partial[Ag, :, :, :, g]   # sum over A != A_g
    ddw[Ag, :, :, :, g] = -sumA_ne          # A=A_g row (corner overwritten below)
    ddw[:, :, Ag, :, g] = -sumB_ne          # B=A_g column
    ddw[Ag, :, Ag, :, g] = sumB_ne.sum(axis=0) - sumB_ne[Ag]   # corner: sum_{A!=A_g, B!=A_g}

### 验证

将解析二阶导数 `ddw` 与数值参考 `ddw_ref` 比较。`ddw_ref` 为 PySCF `get_dweight_dA` (一阶解析导数) 的二阶中心差分，其误差主要来自有限差分截断 $\sim h^2$；在远离原子核、$|s_{AB}|$ 较小以致 $w_g$ 高阶导数较大的格点处，该截断误差更显著。因此中位数达机器精度，个别格点处偏差由数值参考本身的差分误差主导；同时检验 `ddw` 的 $(A,t)\leftrightarrow(B,s)$ 对称性。

In [22]:
diff = np.abs(ddw - ddw_ref)
print(f"median|diff| = {np.median(diff):.4e}")
print(f"max|diff|    = {np.max(diff):.4e}  (far grids: FD truncation ~h^2 in ddw_ref)")
print(f"99.9p|diff|  = {np.percentile(diff, 99.9):.4e}")
print(f"symmetry max|ddw - ddw.T(A<->B)| = {np.max(np.abs(ddw - ddw.transpose(2, 3, 0, 1, 4))):.4e}")
print(f"max|ddw|={np.max(np.abs(ddw)):.4e}  max|ddw_ref|={np.max(np.abs(ddw_ref)):.4e}")
print(f"\nrtol=3e-4, atol=5e-5: {np.allclose(ddw, ddw_ref, rtol=3e-4, atol=5e-5)}")
print(f"rtol=1e-4, atol=1e-6: {np.allclose(ddw, ddw_ref, rtol=1e-4, atol=1e-6)}")

# spot-check a few (A, t, B, s) blocks
for (A, a, B, b, lab) in [(0,0,0,0,"0x0x"), (0,0,0,1,"0x0y"), (0,0,1,0,"0x1x"), (0,0,1,1,"0x1y")]:
    d = np.abs(ddw[A, a, B, b] - ddw_ref[A, a, B, b])
    print(f"  {lab}: max|ddw|={np.max(np.abs(ddw[A,a,B,b])):.4e} "
          f"median|diff|={np.median(d):.4e} max|diff|={np.max(d):.4e}")

median|diff| = 1.3685e-14
max|diff|    = 1.5023e-06  (far grids: FD truncation ~h^2 in ddw_ref)
99.9p|diff|  = 2.5387e-08
symmetry max|ddw - ddw.T(A<->B)| = 1.4211e-14
max|ddw|=1.1543e+02  max|ddw_ref|=1.1543e+02

rtol=3e-4, atol=5e-5: True
rtol=1e-4, atol=1e-6: True
  0x0x: max|ddw|=8.6315e+01 median|diff|=6.0545e-13 max|diff|=1.0921e-06
  0x0y: max|ddw|=3.6751e+01 median|diff|=3.0778e-13 max|diff|=2.5697e-07
  0x1x: max|ddw|=1.0906e+01 median|diff|=5.9025e-14 max|diff|=3.0129e-08
  0x1y: max|ddw|=3.0367e+01 median|diff|=3.6930e-14 max|diff|=1.3881e-07
